In [1]:
# installing libaries
!pip install pandas scikit-learn kagglehub joblib numpy matplotlib mlflow json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [2]:
# loading libraries

# data preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import mlflow


# modelling
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score
from scipy.stats import randint

# visualization
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt

# save model
import joblib
import json

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\timio\.cache\kagglehub\datasets\yasserh\titanic-dataset\versions\1


In [4]:
# reading the data
df = pd.read_csv(f"{path}//Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### Data Preprocessing

In [6]:
df_copy = df.copy()

# Dropping columns - Cabin (a lot of missing data), Ticket
df_copy = df_copy.drop(['Name', 'Cabin', 'Ticket', 'Fare'], axis=1)

# Dropping empty rows
df_copy = df_copy.dropna(axis=0)


df_copy.head()


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,1,0,3,male,22.0,1,0,S
1,2,1,1,female,38.0,1,0,C
2,3,1,3,female,26.0,0,0,S
3,4,1,1,female,35.0,1,0,S
4,5,0,3,male,35.0,0,0,S


In [7]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Sex          712 non-null    object 
 4   Age          712 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Embarked     712 non-null    object 
dtypes: float64(1), int64(5), object(2)
memory usage: 50.1+ KB


Data loss: ~10%

#### Encoding object-type columns

In [8]:
# Using Label Encoder from sklearn
label_encoder = LabelEncoder()
# only for object-type columns
for column in df_copy.columns:
    if df.dtypes[column] == 'object':
        df_copy[column] = label_encoder.fit_transform(df_copy[column])

df_copy.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,1,0,3,1,22.0,1,0,2
1,2,1,1,0,38.0,1,0,0
2,3,1,3,0,26.0,0,0,2
3,4,1,1,0,35.0,1,0,2
4,5,0,3,1,35.0,0,0,2


In [9]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Sex          712 non-null    int64  
 4   Age          712 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Embarked     712 non-null    int64  
dtypes: float64(1), int64(7)
memory usage: 50.1 KB


### Model Training

In [10]:
# setting up mlflow
mlflow.set_experiment("titanic-survivor-predictor")

<Experiment: artifact_location='file:c:/Users/timio/Desktop/projects/titanic-mlops/mlruns/1', creation_time=1776598980071, experiment_id='1', last_update_time=1776598980071, lifecycle_stage='active', name='titanic-survivor-predictor', tags={}, trace_location=None, workspace='default'>

#### Splitting data

In [11]:
# Splitting the data into features (X) and target (y)
X = df_copy.drop(['Survived', 'PassengerId'], axis=1)
y = df_copy['Survived']

# Splitting the data into train and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale the features using Standard Scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### Defining hyperparameters

In [12]:
# Defining model hyperparameters
k = 9   # number of neighbors
params = {
    'k': k,
    'features': list(X.columns)
}

In [13]:
# # Training the model with MLflow Autologging

# # Enabling autologging for scikit-learn
# mlflow.sklearn.autolog()

# # Training the model
# knn = KNeighborsClassifier(n_neighbors=3)
# knn.fit(X_train, y_train)

In [14]:
# # Predictions
# y_pred = knn.predict(X_test)

# # Evaluating the model
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy}")

#### Logging the model

In [15]:
# Manual logging of model

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    # Predict on the test set
    y_pred = knn.predict(X_test)

    # Compute and log the loss metric
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    # Log the model (saving the trained model)
    model_info = mlflow.sklearn.log_model(sk_model=knn, name="titanic_knn_model")

    # Reminder for purpose of run
    mlflow.set_tag("Training Info", "Basic KNN model for titanic data")


2026/04/20 23:22:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


#### Loading the model back for inference

In [16]:
# Loading the trained model
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

predictions = loaded_model.predict(X_test)

# Feature names
titanic_feature_names = X.columns

result = pd.DataFrame(X_test, columns=titanic_feature_names)
result["actual_class"] = y_test.values
result["predicted_class"] = predictions

result.head()

,Pclass,Sex,Age,SibSp,Parch,Embarked,actual_class,predicted_class
0,0.930236,0.742564,-0.602897,-0.545029,-0.494850,0.495868,0,0
1,0.930236,-1.346685,-0.805612,-0.545029,0.707274,-2.159424,0,1
2,0.930236,-1.346685,-0.940755,5.016944,1.909397,0.495868,0,0
3,-0.260657,-1.346685,-0.129897,-0.545029,-0.494850,0.495868,1,1
4,0.930236,-1.346685,-1.413755,2.792155,1.909397,0.495868,0,0


#### Evaluating different values of k

In [17]:
k_values = [i for i in range(1, 31)]

for k in k_values:
    with mlflow.start_run():
        mlflow.log_param("k", k)

        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)

        # Predict on the test set
        y_pred = knn.predict(X_test)

        # Compute and log the loss metric
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # Log the model (saving the trained model)
        mlflow.sklearn.log_model(sk_model=knn, name="titanic_knn_model")
        

2026/04/20 23:23:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 23:23:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 23:23:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

### Saving the model

In [19]:
# Saving model
best_k = 20     # best k from training

# best model
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train, y_train)

filename = 'knn_model.pkl'
joblib.dump(best_model, filename)


['knn_model.pkl']

In [22]:
# Saving column order for inference
columns = list(X.columns)
# columns

# writing to json file - used for API
with open('columns.json', 'w') as file:
    json.dump(columns, file)

print(f"Saved: {columns}")

Saved: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Embarked']
